In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: one protected common direction, same START_FINAL and .10 step

**Run all first saves the bit29 protection gradient and stops normally.**
When PAUSED_FOR_SESSION_RESTART appears, delete/disconnect the Colab runtime,
reconnect to a new GPU runtime, and Run all again. Keep the same Drive folder.
The one-time stop is recorded; a later Run all skips the completed gradient.
Simply starting another child process is not evidence of a new Colab runtime.

Original BASE/mask/key/message/readout and radius cap 1 are retained.
START_FINAL is the saved CONT_FINAL used by Six-Bit-Final. Reuse its six gradients
for bits [4,12,18,20,27,30]. Reuse BASE, START_FINAL and TANGENT RGB/readouts from
the completed Tangent-Common task. There are no replacement decodes of old views.
Input locations and source branch are editable. No source/identity checks are required.

Four serial independent VJPs at START, in order:
* positive margin29 directly;
* B derivative: -mean25 log((m_j-.02)/(m_j(START)-.02)), excluding bit29;
* G directly: c0-max(other correlations), requiring a unique active wrong offset;
* Q: full RGB MSE relative to saved original BASE.
B uses fixed START coefficients -1/[25*(m_j(START)-.02)] to differentiate once.
This is aggregate protection, not individual protection for all25 bits. A log-domain
failure at the endpoint is reported without epsilon, clipping or dropping the bit.
Q uses the existing VAE float RGB mapping clamp[0,1], with FP64 residual arithmetic;
residuals are not clipped, quantized or encoded. Q relative to START is not used.

All gradients use the fixed mask and START-minus-BASE tangent plane. A CPU SOCP in
their at-most-ten-dimensional span maximizes the minimum raw six-bit margin slope,
with margin29/G nondecreasing and B/Q nonincreasing predictions, and norm<=1.
Six target gradients are never individually normalized. Protection rows can be
positively scaled for numerical conditioning. Zero is always feasible; unresolved
solver status is not method infeasibility. No automatic constraint relaxation.
Only a numerically clear positive result defines a unit direction and one .10 step.
Candidate=FP32(FP64(START_FINAL)+.10*direction), no endpoint projection, second step,
step scan, rollback, radius fill, cap increase or new source point. Candidate radius
is approximately .959667. Actual FP32 increment predictions and radius are saved.

Fixed report views BASE / START_FINAL / TANGENT / PROTECTED retain all32 margins,
22 correlations and G: 352 windows/5632 chips/220 metrics including three reused
views. The four gradient probe reads have a separate denominator; they are not
independent samples. Failed, undefined and unexecuted slots remain in the report.
If no clear positive direction exists, PROTECTED remains NOT_RUN and the result
package is COMPLETE_NO_CANDIDATE. No extra model call tries to improve that result.

Nominal new work without a pause: **1 load + 4 gradient decodes / 4 VJPs +
1 candidate ordinary decode**, 184 gradient-forward and nominal replay chunks,
46 ordinary chunks. The planned restart adds one load: expected cumulative totals
after completion are 2 loads / 5 decodes / 4 VJPs when nothing else is interrupted.
If no candidate is defined, ordinary decode is skipped. Physical attempts can
exceed these budgets only when unfinished units must be recovered; all are counted.

Uses the already-run checkpoint_decode and activation-spool path. Every60 seconds
stores last committed progress, not the live autograd graph. Inputs, each complete
gradient, direction, candidate latent and decoded RGB commit immediately on Drive.
An interrupted VJP/decode restarts only that unfinished unit; complete units are
skipped. The activation spool is temporary local storage, not a durable graph.
Complete report recovery needs no VAE/CUDA. Call receipts and failures accumulate.

Stable task: MyDrive/Video-WM/Local-Joint-State-Payload-V1-Protected-Common/
resumable-protected-common-v1/. Do not change this output for the restart.
Four-view report includes six-bit gain retention versus old TANGENT, all32 bit
changes, bit29 versus mu=.02, other25 losses/domain failures, G versus gamma=.01,
full frame/temporal quality versus BASE and START, peak pixels and neighborhoods.
BASE is the already-watermarked JOINT BASE, not OFF. MSE alone is not quality PASS.
First-order constraints can fail after real decoding; retain and report those outcomes.
This one-point known-message diagnostic is not blind decoding, codec survival,
FPR, trajectory/CFG/native causality, generalization or scientific PASS.

Delivery checks are CPU/fake/static only; GPU execution is yours. Return the entire
stable task directory including failures, attempts and checkpoint sidecars.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-protected-common-v1', 'source_run': 'resumable-six-bit-final-v1', 'source_arm': 'CONT_FINAL', 'step_size': 0.1, 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'device': 'cuda', 'inputs': {'six_bit_inputs': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Six-Bit-Final/resumable-six-bit-final-v1/run/inputs/fixed_inputs.pt', 'gradient_units': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Six-Bit-Final/resumable-six-bit-final-v1/run/units', 'tangent_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Tangent-Common/resumable-tangent-common-v1/run'}}
EXPECTED_COST = {'vae_load': 1, 'decode': 5, 'decoder_vjp': 4, 'ordinary_decode': 1, 'encode': 0, 'dit': 0, 'native': 0, 'scheduler': 0, 'codec': 0, 'ordinary_chunks': 46, 'gradient_forward_chunks': 184, 'nominal_replay_chunks': 184, 'windows': 352, 'chips': 5632, 'metrics': 220, 'gradient_probe_windows': 352, 'gradient_probe_chips': 5632, 'gradient_probe_metrics': 220, 'reused_view_count': 3, 'new_view_count': 1, 'planned_pause_extra_load': 1}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Protected-Common')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-protected-common-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Protected-Common-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_protected_common_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('BASE', 'START_FINAL', 'TANGENT', 'PROTECTED')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed PROTECTED-COMMON output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_protected_common_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_protected_common_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    NEEDS_SOLVER = not (RUN_OUTPUT / 'direction_unit.pt').is_file()
    if NEEDS_SOLVER:
        probe_lines += ['import cvxpy, clarabel']
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for unfinished decode/VJP'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_SOLVER: packages += ['cvxpy', 'clarabel']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_protected_common_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_PROTECTED_COMMON', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_protected_common_v1 import finalize_interrupted
    # Takeover is report-only and follows child/process-group reap in logged().
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] not in ('COMPLETE', 'COMPLETE_NO_CANDIDATE', 'PAUSED_FOR_SESSION_RESTART'):
        raise RuntimeError('PROTECTED-COMMON engineering interruption; retain entire output: ' + str(OUTPUT))
    if result['status'] == 'PAUSED_FOR_SESSION_RESTART':
        print('The first new gradient is saved. Delete/disconnect this Colab runtime, reconnect, and Run all again. Completed units will be skipped.', flush=True)
    fixed_slots.update(status=result['status'], result='run/result.json')
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_PROTECTED_COMMON', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Engineering status:', result['status'])
print('Nominal budget:', result['expected_cost'])
print('Cumulative physical calls:', result['counts'])
print('This Run all:', result.get('current_attempt_counts', {}))
print('Ordinary chunks:', result['ordinary_chunks'])
for name,row in result['gradient_units'].items():
    print('Protection unit:', name, row['status'])
for name,replay in result['replay'].items():
    print('Gradient ledger:', name, replay['counts'], replay.get('boundary_storage', {}))
print('Direction:', result['steps']['direction'])
print('Step:', result['steps']['step'])
for name,row in result['views'].items():
    print(name,row['status'],'G=',row.get('state_gap'),'positive bits=',
          sum(x is not None and x>0 for x in row.get('margins',[])),'/32','reason=',row.get('reason'))
if result['status'] == 'PAUSED_FOR_SESSION_RESTART':
    print('First new gradient is saved. Delete/disconnect runtime, reconnect, then Run all again with this same task folder.')
else:
    comparison = json.loads((RUN_OUTPUT / 'comparison.json').read_text())
    for bit in (4,12,18,20,27,30):
        row=comparison['bits_32'][bit]
        print('bit',bit,'margins=',row['margins'],'gain=',row['protected_gain'],'retained=',row['retained_tangent_gain_ratio'])
    print('Protection observations:',comparison['protections'])
    print('Quality versus BASE:',comparison['quality_vs_BASE'])
    print('Resume observation:',result.get('resume_observation'))
print(result['ceiling'])
print('Return the entire output folder, including failures:', OUTPUT)